# Credit Risk Economic Capital Demo

**Counterparty Credit Risk & Economic Capital Simulator**  
Ajayvir Khara | Passed FRM Part I and Part II | January 2026

This notebook demonstrates a complete counterparty credit risk & economic capital workflow:

- Generation / loading of stylized portfolio of counterparties
- Simulation of future exposure profiles (EE, PFE, EPE) under CSA
- Computation of Expected Loss (EL), Unexpected Loss (UL), Economic Capital (EC)
- **Enhanced** Wrong-Way Risk (WWR) adjustment (affects both EL and UL)
- Marginal / Euler allocation of portfolio economic capital
- Regulatory-style Excel report generation

**Expected runtime**: ~45–150 seconds depending on number of paths & counterparties

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import norm
import sys
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Styling
plt.style.use("seaborn-v0_8-pastel")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (14, 7)
%matplotlib inline

# Project root detection
notebook_dir = Path.cwd().resolve()
possible_roots = [
    notebook_dir,
    notebook_dir.parent,
    notebook_dir.parent.parent,
    notebook_dir.parent.parent.parent,
]
project_root = None
for p in possible_roots:
    if (p / "econ_capital").is_dir():
        project_root = p
        break
if project_root is None:
    raise RuntimeError("Could not find project root. Set project_root manually.")

print("Project root:", project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from econ_capital.credit_risk import (
    Trade,
    NettingSet,
    CSA,
    ExposureEngine,
    compute_counterparty_risk_profiles,
    aggregate_credit_losses,
    DEFAULT_CONFIG,
    simulate_credit_factors,
)
from econ_capital.credit_risk.demo_exposure import _simulate_sp500_paths
from econ_capital.credit_risk.creditrisk_reporting import generate_creditrisk_report

## 1. Configuration & Parameters

In [ ]:
CONFIG = DEFAULT_CONFIG.copy()

PARAMS = {
    "n_paths": 8000,
    "horizon_years": 1.0,
    "n_time_steps": 13,
    "confidence_level": 0.999,
    "pfe_quantile": 0.975,
    "alpha_factor": 1.4,
    "wwr_sensitivity": 0.35,
    "portfolio_correlation": 0.30,
    "seed": 42,
}

pd.Series(PARAMS).to_frame("Value")

## 2. Generate/Load Portfolio

In [ ]:
from econ_capital.credit_risk.generate_portfolio import generate_portfolio_csv

DATA_DIR = project_root / "econ_capital" / "credit_risk" / "data"
CSV_PATH = DATA_DIR / "counterparty_exposures.csv"

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    print("Generating portfolio...")
    generate_portfolio_csv(filename=CSV_PATH.name, n_cptys=60, seed=PARAMS["seed"])

print("Using portfolio:", CSV_PATH.name)

## 3. Market Factor Simulation

In [ ]:
times = np.linspace(0, PARAMS["horizon_years"], PARAMS["n_time_steps"])

market_paths = _simulate_sp500_paths(
    n_paths=PARAMS["n_paths"],
    times=times,
    s0=100.0,
    mu=0.0,
    sigma=0.25,
    seed=PARAMS["seed"],
)

print("Market paths generated:", {k: v.shape for k, v in market_paths.items()})

## 4. Exposure Profile (Stylised)

In [ ]:
trades = [
    Trade("Vanilla IRS", "SP500", w=0.85, gamma=0.0),
    Trade("Equity Forward", "SP500", w=0.40, gamma=0.004),
]

csa = CSA(threshold=5_000_000, mta=2_000_000, im=3_500_000, vm_calls_per_day=1)

netting_set = NettingSet("DEMO_PORT", trades=trades, csa=csa)

engine = ExposureEngine(
    netting_set=netting_set,
    market_paths=market_paths,
    times=times,
    n_paths=PARAMS["n_paths"],
    pfe_quantile=PARAMS["pfe_quantile"],
    alpha_factor=PARAMS["alpha_factor"],
)

exposure_paths, exposure_summary = engine.compute_exposure_profile()

display(exposure_summary.round(2))

## 5. Counterparty Risk Profiles + Enhanced WWR

In [ ]:
demo_ead = float(exposure_summary["EAD_final"].iloc[-1])

portfolio = pd.read_csv(CSV_PATH)
portfolio = portfolio.rename(columns={"pd_annual": "PD", "value": "EAD"})

if "LGD" not in portfolio.columns:
    portfolio["LGD"] = 0.45

portfolio["EAD_final"] = demo_ead * np.random.uniform(0.4, 1.6, len(portfolio))

risk_df = compute_counterparty_risk_profiles(portfolio.to_dict("records"))

# ────────────────────────────────────────────────────────────────
# Path-wise Wrong-Way Risk
print("Applying path-wise WWR adjustment (strong tail effect)...\n")

credit_factors = simulate_credit_factors(
    n_paths=PARAMS["n_paths"],
    n_steps=len(risk_df),
    corr=0.25,  # or use a parameter if you have one
    seed=PARAMS["seed"] + 1,
)

n_paths, n_cpty = credit_factors.shape

# Scale EL per path (adverse shocks only)
el_paths = risk_df["EL"].values[None, :] * (
    1 + PARAMS["wwr_sensitivity"] * 3.5 * np.maximum(credit_factors, 0)
)

# Scale UL proxy per path (stronger multiplier for tail realism)
ul_paths = risk_df["UL"].values[None, :] * (
    1 + PARAMS["wwr_sensitivity"] * 5.0 * np.maximum(credit_factors, 0)
)

# Portfolio loss per path (stylized – EL + scaled UL contribution)
portfolio_losses = el_paths.sum(axis=1) + 3.09 * ul_paths.sum(axis=1) * 0.7

# WWR-adjusted EC at 99.9% quantile
EC_wwr_pathwise = np.quantile(portfolio_losses, 0.999)

risk_df["EL_WWR"] = el_paths.mean(axis=0)  # mean EL across paths
risk_df["UL_WWR"] = ul_paths.mean(axis=0)  # mean UL scaling across paths

print("Created compatibility columns for aggregation:")
print(" - 'EL_WWR' shape:", risk_df["EL_WWR"].shape)
print(" - 'UL_WWR' shape:", risk_df["UL_WWR"].shape)

# Comparison with base (non-WWR) EC
base_ec = risk_df["EL"].sum() + norm.ppf(0.999) * risk_df["UL"].sum() * 0.7

print(f"Base EC (no WWR):           £{base_ec:,.0f}")
print(f"Path-wise WWR EC:           £{EC_wwr_pathwise:,.0f}")
print(f"Increase due to WWR:        {EC_wwr_pathwise / base_ec - 1:.1%}")

# Store for later use / reporting
risk_df["EC_WWR_pathwise"] = EC_wwr_pathwise / n_cpty  # rough per-counterparty share

## 6. Portfolio Economic Capital + Allocation

In [ ]:
n = len(risk_df)
corr_matrix = np.full((n, n), PARAMS["portfolio_correlation"])
np.fill_diagonal(corr_matrix, 1.0)

EL_tot, UL_tot, EC_tot, alloc = aggregate_credit_losses(
    el=risk_df["EL_WWR"].values,
    ul=risk_df["UL_WWR"].values,
    corr=corr_matrix,
    confidence=PARAMS["confidence_level"],
)

risk_df["EC_Marginal"] = alloc

print(f"Portfolio EL (WWR):      £{EL_tot:,.0f}")
print(f"Portfolio UL (WWR):      £{UL_tot:,.0f}")
print(f"Portfolio EC (99.9%):    £{EC_tot:,.0f}")

## 7. Top Contributors Visualization

In [ ]:
top_n = 12
top = risk_df.sort_values("EC_Marginal", ascending=False).head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(
    data=top,
    x="EC_Marginal",
    y="counterparty",
    hue="counterparty",
    palette="Reds_r",
    legend=False,
    ax=ax,
)

ax.set_title(f"Top {top_n} Contributors to Economic Capital (WWR-adjusted)")
ax.set_xlabel("Allocated Economic Capital (£)")
ax.set_ylabel("Counterparty")
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

## 8. Generate Regulatory-style Report

In [ ]:
import os
import glob
import time

report_data = {
    "EC_total": EC_tot,
    "EL_total": EL_tot,
    "UL_total": UL_tot,
    "capital_breakdown": risk_df.set_index("counterparty")["EC_Marginal"],
    "full_data": risk_df,
}

report_path = generate_creditrisk_report(
    config=CONFIG,
    engine=None,
    results=report_data,
    output_dir=str(project_root / "econ_capital" / "credit_risk" / "reports"),
)

print(f"\nReport generated:\n{report_path}")

time.sleep(1)  # give file system a moment to finish writing

report_pattern = str(
    project_root
    / "econ_capital"
    / "credit_risk"
    / "reports"
    / "CreditRisk_EC_Report_*.xlsx"
)
reports = glob.glob(report_pattern)

if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"Opening latest report: {os.path.basename(latest_report)}")
    os.startfile(latest_report)  # ← Windows only
else:
    print("No report found in the reports directory.")

## Next steps / experiments

- Try different `wwr_sensitivity` values (0.1–0.8)
- Change `portfolio_correlation` (0.1–0.7) and observe diversification
- Increase `n_paths` to 20k–50k for more stable results
- Experiment with different CSA configurations (threshold, IM, call frequency)
- Compare daily vs weekly margin calls on EAD